# grad-tracking-global-toggle — ex3: set_grad_enabled(mode) — value-taking context manager

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-tracking-global-toggle`. Running the final beacon cell reports progress against the `Backprop: Grad-tracking toggle` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad-tracking toggle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-tracking-global-toggle`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-tracking-global-toggle"
DD_SUBTOPIC = "Backprop: Grad-tracking toggle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `set_grad_enabled(mode)` — explicit-value setter, not just toggle off

Ex1 had `NoGrad()` (forces off, restores). Ex2 had `no_grad` decorator (forces off, restores even on exception). PyTorch ALSO ships `torch.set_grad_enabled(mode: bool)` — a context manager that sets the toggle to an EXPLICIT value (`True` OR `False`), and restores the previous value on exit. This is what you reach for inside a `no_grad` block when you want to TEMPORARILY re-enable grad for one inner step.

```python
with set_grad_enabled(False):     # outer: off
    assert grad_tracking_enabled is False
    with set_grad_enabled(True):  # inner: explicitly on again
        assert grad_tracking_enabled is True
    assert grad_tracking_enabled is False  # back to outer
assert grad_tracking_enabled is True       # back to global default
```

**Why a value-taking ctx manager is necessary.** `NoGrad` can only force off. If you've already disabled grad via `no_grad` and want one inner step to re-enable it (e.g. compute a regularizer gradient during a no-grad eval loop), you need a setter that takes `True`. Same restore semantics as `NoGrad`, but the entry value is a parameter.

**Stack discipline.** Each `__enter__` snapshots the CURRENT value, sets the new one. Each `__exit__` restores the snapshot. Arbitrary nesting works because every frame's snapshot is independent.

### Exercise 3 — set_grad_enabled(mode) — value-taking context manager

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the snapshot-and-restore context-manager pattern to a setter that takes an EXPLICIT bool value (not just 'force off'), so nested set_grad_enabled(True) can re-enable grad inside an outer no-grad block.
> Keywords: set_grad_enabled, context-manager, value-setter, nesting
> ```

**KCs targeted:** `grad-tracking-global-toggle`, `context-manager-restores-snapshot`

Implement `ex3_set_grad_enabled()` returning a class `SetGradEnabled` and a 'getter' fn `get_state()`.

The class must:

1. `__init__(self, mode: bool)` — store `mode` as `self.mode`.
2. `__enter__(self)` — snapshot the CURRENT module-level `grad_tracking_enabled` into `self.prev`, then set the module global to `self.mode`. Return `self`.
3. `__exit__(self, exc_type, exc_val, tb)` — restore the snapshot. Return `False` so exceptions propagate.

Module state:

- Define `grad_tracking_enabled = True` at module/global scope BEFORE the class.
- The class MUST read/write the GLOBAL via `globals()` or explicit `global grad_tracking_enabled` — never snapshot it into the closure.

Helper:

- `get_state()` returns the current `grad_tracking_enabled` (so the test can poll it without depending on the test's own scope).

Return: `{'SetGradEnabled': SetGradEnabled, 'get_state': get_state, 'reset': reset}` where `reset()` sets the global back to `True`.

In [ ]:
def ex3_set_grad_enabled():
    # Module-level state lives in a dict so the closure can mutate it.
    state = {'grad_tracking_enabled': True}

    class SetGradEnabled:
        def __init__(self, mode):
            self.mode = bool(mode)
            self.prev = None
        def __enter__(self):
            self.prev = state['grad_tracking_enabled']
            state['grad_tracking_enabled'] = self.mode
            return self
        def __exit__(self, exc_type, exc_val, tb):
            state['grad_tracking_enabled'] = self.prev
            return False

    def get_state():
        return state['grad_tracking_enabled']

    def reset():
        state['grad_tracking_enabled'] = True

    return {'SetGradEnabled': SetGradEnabled, 'get_state': get_state,
            'reset': reset}


<details><summary>Solution</summary>

```python
def ex3_set_grad_enabled():
    # Module-level state lives in a dict so the closure can mutate it.
    state = {'grad_tracking_enabled': True}

    class SetGradEnabled:
        def __init__(self, mode):
            self.mode = bool(mode)
            self.prev = None
        def __enter__(self):
            self.prev = state['grad_tracking_enabled']
            state['grad_tracking_enabled'] = self.mode
            return self
        def __exit__(self, exc_type, exc_val, tb):
            state['grad_tracking_enabled'] = self.prev
            return False

    def get_state():
        return state['grad_tracking_enabled']

    def reset():
        state['grad_tracking_enabled'] = True

    return {'SetGradEnabled': SetGradEnabled, 'get_state': get_state,
            'reset': reset}
```

**Per-instance `self.prev` is the snapshot key.** Each context instance captures the value AT THE MOMENT OF `__enter__` — not at construction. That's why you can write `ctx_a = SetGradEnabled(False)` and only have it take effect inside `with ctx_a:`.

**Why a state dict instead of a real module global.** Python's `global` declarations work inside a real module but not inside a function closure. Wrapping the toggle in a dict gives us a single mutable cell that all inner functions can share without import-time gymnastics. The behaviour is identical from the outside.

**`return False` from `__exit__` re-raises exceptions.** Returning truthy would suppress the exception — wrong for a grad-tracking utility, which has no business swallowing errors.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()